In [1]:
import glob, numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: check the attached datasets (and the aes_dataset version)"
    return hits[0]

aes = (pd.read_csv(find("aes_prepared.csv"))
       .merge(pd.read_csv(find("aes_features.csv")), on="essay_id")
       .merge(pd.read_csv(find("aes_languagetool.csv")), on="essay_id")
       .merge(pd.read_csv(find("aes_cola.csv")), on="essay_id")
       .merge(pd.read_csv(find("oof_aes_deberta.csv"))[["essay_id", "oof_score"]], on="essay_id"))
feature_cols = [c for c in aes.columns if c not in {"essay_id", "text", "score", "fold", "word_count", "oof_score"}]
y, fold = aes["score"].values, aes["fold"].values
print(len(feature_cols), "features |", aes.shape)

def qwk(y, p): return cohen_kappa_score(y, p, weights="quadratic")
def labels(p, cuts): return 1 + np.searchsorted(np.sort(cuts), p)
def fit_cuts(p, y):
    r = minimize(lambda c: -qwk(y, labels(p, c)), [1.5, 2.5, 3.5, 4.5, 5.5], method="Nelder-Mead",
                 options={"maxiter": 400, "xatol": 1e-3, "fatol": 1e-6})
    return np.sort(r.x)
def nested_qwk(p):
    pred = np.zeros(len(y), dtype=int)
    for f in range(5):
        pred[fold == f] = labels(p[fold == f], fit_cuts(p[fold != f], y[fold != f]))
    return qwk(y, pred)
rmse = lambda p: float(np.sqrt(np.mean((p - y) ** 2)))

params = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=30, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, verbose=-1)
def lgb_oof(cols):
    oof = np.zeros(len(aes))
    for f in range(5):
        tr, va = fold != f, fold == f
        oof[va] = lgb.LGBMRegressor(**params).fit(aes.loc[tr, cols], y[tr]).predict(aes.loc[va, cols])
    return oof

oof_deb   = aes["oof_score"].values
oof_feat  = lgb_oof(feature_cols)
oof_stack = lgb_oof(feature_cols + ["oof_score"])
w = min(np.linspace(0, 1, 21), key=lambda w: rmse(w * oof_deb + (1 - w) * oof_feat))
oof_blend = w * oof_deb + (1 - w) * oof_feat
print(f"blend weight on DeBERTa: {w:.2f}")

cands = {"A. DeBERTa": oof_deb, "B. LightGBM features + CoLA": oof_feat,
         "C. Blend of A and B": oof_blend, "D. LightGBM stack (features + DeBERTa)": oof_stack}
default = [1.5, 2.5, 3.5, 4.5, 5.5]
table = pd.DataFrame({name: {"RMSE": rmse(p), "QWK plain rounding": qwk(y, labels(p, default)),
                             "QWK tuned cut-points (nested)": nested_qwk(p)} for name, p in cands.items()}).T
print("\n" + table.round(4).to_string())

pd.DataFrame({"essay_id": aes["essay_id"], "fold": fold, "score": y, "deberta": oof_deb,
              "lgbm": oof_feat, "blend": oof_blend, "stack": oof_stack}).to_csv("/kaggle/working/aes_stacking_oof.csv", index=False)

64 features | (17307, 70)
blend weight on DeBERTa: 0.65

                                          RMSE  QWK plain rounding  QWK tuned cut-points (nested)
A. DeBERTa                              0.5658              0.8060                         0.8110
B. LightGBM features + CoLA             0.6164              0.7574                         0.7767
C. Blend of A and B                     0.5488              0.8065                         0.8221
D. LightGBM stack (features + DeBERTa)  0.5457              0.8138                         0.8234


In [2]:
import json
cuts = fit_cuts(oof_blend, y)
final_pred = labels(oof_blend, cuts)
print("Cut-points:", np.round(cuts, 3), "| QWK with these cut-points (in-sample):", round(qwk(y, final_pred), 4))
print("exact match: %.1f%% | within one point: %.1f%%" % ((final_pred == y).mean() * 100, (np.abs(final_pred - y) <= 1).mean() * 100))
print("\ntrue (rows) vs predicted (columns):")
print(pd.crosstab(y, final_pred, rownames=["true"], colnames=["pred"]).to_string())

terc = pd.qcut(aes["n_words"], 3, labels=["short", "medium", "long"])
print("\nBy essay length:")
for name in ["short", "medium", "long"]:
    m = (terc == name).values
    print(f"  {name:<7} QWK {qwk(y[m], final_pred[m]):.3f} | mean error {np.mean(oof_blend[m] - y[m]):+.3f}")
print("correlation between the error and essay length:", round(np.corrcoef(oof_blend - y, aes["n_words"])[0, 1], 3))

final_lgb = lgb.LGBMRegressor(**params).fit(aes[feature_cols], y)
final_lgb.booster_.save_model("/kaggle/working/lgbm_aes_features.txt")
json.dump({"deberta_weight": float(w), "cuts": [float(c) for c in cuts], "feature_cols": feature_cols},
          open("/kaggle/working/aes_blend_config.json", "w"), indent=1)
imp = pd.Series(final_lgb.booster_.feature_importance("gain"), index=feature_cols)
print("\nTop 12 features in the LightGBM part (share of gain, %):")
print((imp / imp.sum() * 100).sort_values(ascending=False).head(12).round(1).to_string())

Cut-points: [1.696 2.543 3.383 4.225 5.104] | QWK with these cut-points (in-sample): 0.8239
exact match: 64.8% | within one point: 98.6%

true (rows) vs predicted (columns):
pred    1     2     3     4    5   6
true                                
1     756   438    53     5    0   0
2     405  3337   916    64    1   0
3      15  1138  3875  1199   53   0
4       0    18   768  2566  568   6
5       0     0    11   293  643  23
6       0     0     0     9  112  35

By essay length:
  short   QWK 0.629 | mean error -0.003
  medium  QWK 0.638 | mean error +0.009
  long    QWK 0.676 | mean error +0.005
correlation between the error and essay length: -0.062

Top 12 features in the LightGBM part (share of gain, %):
n_chars             44.1
n_words             14.6
ttr                  2.7
lt_nontypo_total     2.4
cola_mean            2.4
cola_p25             1.7
lt_typos             1.7
pos_aux              1.4
digit_ratio          1.4
n_unique_words       1.3
pos_propn            1.2
avg_